### Load Data

In [ ]:
import pandas as pd
from pandas import DataFrame

df = pd.read_csv(r"data/housing.csv")

### Understand Data

In [ ]:
df.head()
df.info()
df.value_counts()
df.describe()

### Visualize Data
#### Histogram

In [ ]:
import matplotlib.pyplot as plt

df.hist(bins=50, figsize=(16,8))
plt.show()

#### Scatter Plot

In [ ]:
import seaborn as sns

num_df = df[["housing_median_age", "total_rooms", "median_income", "median_house_value"]]

sns.pairplot(num_df)
plt.show()

#### Correlation

In [ ]:
df.plot(kind="scatter", x="median_income", y="median_house_value")
plt.show()

### Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size=0.2, random_state=42)
test

### Stratified Split

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

splitter = StratifiedShuffleSplit(n_splits=10, test_size=0.2, random_state=42)
splits = splitter.split(df, df["ocean_proximity"])
train, test = list(splits)[0]
test

In [ ]:
train, test = train_test_split(df, test_size = 0.2, random_state=42, stratify=df["ocean_proximity"])
train

### Data Prep

### Clean Data

In [ ]:
# df.dropna()
# df.dropna("column", axis=1)
# df.fillna(df.mean())

### Imputer

In [ ]:
from sklearn.impute import SimpleImputer

house_value = train[["median_house_value"]]

imputer = SimpleImputer(strategy="median")
imputer.fit(house_value)
imputer.transform(house_value)

### Encoder

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

encoder = OrdinalEncoder()
proximity = train[["ocean_proximity"]]
encoder.fit(proximity)
encoder.transform(proximity)

# encoder.fit_transform(df)

### Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler
# from sklearn.preprocessing import MinMaxScaler

scaler = StandardScaler()
scaler.fit(train)
scaler.transform(train)

# scaler.fit_transform(tain)

### Pipleline

impute -> encode -> scale

In [ ]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("encode", OrdinalEncoder()),
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])

transformed = pipeline.fit_transform(train)

df_transformed = DataFrame(transformed, columns = train.columns, index = train.index)

### Train

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(df_transformed, df_transformed["median_house_value"])

In [ ]:
predictions = model.predict(df_transformed)

### Evaluate

In [ ]:
from sklearn.metrics import mean_squared_error
mse = mean_squared_error(df_transformed["median_house_value"], predictions)

In [ ]:
from sklearn.model_selection import cross_val_score

res = cross_val_score(model,  
                      df_transfromed, 
                      df_transformed["median_house_value"], 
                      cv= 5, 
                      scroing="root_mean_squared_error")
res

### Persist Model

In [ ]:
import joblib

joblib.dump(model, "lin_reg_housing.pkl")

model = joblib.load("lin_reg_housing.pkl")

# model.predict(test)